In [1]:
import os
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import gp
import math
import glob

# --- CONFIG ---------------------------------------------------------
root_file  = 'EventSelection_Data_10Percent.root'
hist_name  = 'h_Minv_General_Final_1'
coeffs     = [0.00038, 0.041, -0.27, 3.49, -11.11]
neighborhood_rebin = 1
n_restarts = 10

# how many toys per make_table call
N_TOYS = 5000

output_dir = 'ul_tables'
os.makedirs(output_dir, exist_ok=True)

# --------------------------------------------------------------------
def sigma(m, coeffs):
    """Return 2σ mass resolution from polynomial coeffs"""
    return sum(c * m**i for i, c in enumerate(coeffs))


def estimate_background(mass, coeffs, rebin, restarts):
    """
    Fit GP excluding blind ±2σ region, then return
    total mean, total uncertainty, and observed sum in blind.
    """
    sigma_val = sigma(mass, coeffs)
    lower, upper = mass - 3*sigma_val, mass + 3*sigma_val
    blind = (mass - 1.96*sigma_val, mass + 1.96*sigma_val)

    manip = gp._hist.manipulation.rebin_and_limit(rebin, lower, upper)
    model = gp.GaussianProcessModel(
        h=(root_file, hist_name),
        kernel=(gp.kernels.WhiteKernel(noise_level=7e3)
                + gp.kernels.RBF(length_scale=0.016)
                  * gp.kernels.DotProduct(sigma_0=2.5e4)),
        n_restarts_optimizer=restarts,
        blind_range=blind,
        modify_histogram=manip
    )

    X     = model.histogram.axes[0].centers
    vals  = model.histogram.values()

    # train on outside blind region
    mask_train = (X < blind[0]) | (X > blind[1])
    X_train = X[mask_train].reshape(-1,1)
    y_train = vals[mask_train]

    # get sklearn GPR under the hood
    gpr = getattr(model, 'model', getattr(model, '_model', None))
    if gpr is None:
        raise RuntimeError("Cannot find sklearn GPR model on GaussianProcessModel")

    gpr.fit(X_train, y_train)

    # predict inside blind
    mask_blind = (X >= blind[0]) & (X <= blind[1])
    Xb = X[mask_blind].reshape(-1,1)
    mu, std = gpr.predict(Xb, return_std=True)

    pred_sum   = np.sum(mu)
    uncert_sum = np.sqrt(np.sum(std**2))
    obs_sum    = np.sum(vals[mask_blind])
    return pred_sum, uncert_sum, obs_sum


def make_table(masses, coeffs, rebin, restarts, n_toys):
    """
    For an array of masses:
      1. Estimate background pred/unc/obs for each mass
      2. Generate toy UL distributions
      3. Compute ±2σ, ±1σ, median, and observed UL
    Returns:
      (obs_ul, lo2, lo1, med, hi1, hi2)
    """
    # 1) get predicted bg and obs for each mass
    preds = []
    uncerts = []
    obs = []
    for m in masses:
        p, u, o = estimate_background(m, coeffs, rebin, restarts)
        preds.append(p)
        uncerts.append(u)
        obs.append(o)
    preds  = np.array(preds)
    uncerts= np.array(uncerts)
    obs    = np.array(obs, dtype=int)

    # 2) draw toys: smear by uncert -> Poisson
    rng = np.random.default_rng(42)
    toy_counts = rng.poisson(
        rng.normal(loc=preds, scale=uncerts, size=(n_toys, len(masses)))
       )
        

    # 3) compute toy ULs
    toy_ul = gp.limit_setting._single_bin_cls(
        preds[np.newaxis,:],
        toy_counts
    )

    # 4) quantiles
    lo2, lo1, med, hi1, hi2 = np.quantile(
        toy_ul, [0.025, 0.16, 0.5, 0.84, 0.975], axis=0
    )

    # 5) observed UL
    obs_ul = gp.limit_setting._single_bin_cls(preds, obs)

    return obs_ul, lo2, lo1, med, hi1, hi2


# ----------------------------------------------------------
# MAIN SCAN: one file per center mass
# ----------------------------------------------------------
for center in np.arange(0.033, 0.180, 0.001):
    # define fine mass grid in ±3σ around center
    sig = sigma(center, coeffs)
    lb, ub = center - 3*sig, center + 3*sig
    masses = np.round(np.arange(lb, ub+1e-9, 0.001), 3)

    try:
        # compute observed UL and Brazil bands
        obs_ul, lo2, lo1, med, hi1, hi2 = make_table(
            masses, coeffs, neighborhood_rebin, n_restarts, N_TOYS
        )
        # assemble DataFrame
        df = pd.DataFrame({
            'Mass':         masses,
            'Observed UL':  obs_ul,
            'Median UL':    med,
            'Lower 1σ':     lo1,
            'Upper 1σ':     hi1,
            'Lower 2σ':     lo2,
            'Upper 2σ':     hi2
        })
    except Exception as e:
        print(f"make_table failed at center {center:.3f} GeV: {e}")
        # write empty CSV on failure
        df = pd.DataFrame(columns=[
            'Mass', 'Observed UL', 'Median UL',
            'Lower 1σ', 'Upper 1σ', 'Lower 2σ', 'Upper 2σ'
        ])

    # write CSV
    fname = f"UL_{center:.3f}GeV.csv"
    df.to_csv(os.path.join(output_dir, fname), index=False)
    
all_files = glob.glob(os.path.join(output_dir, "UL_*.csv"))
print(f"Combining {len(all_files)} files...")
df_list = [pd.read_csv(f) for f in all_files]
if df_list:
    combined = pd.concat(df_list, ignore_index=True)
    # Quantize Mass and group by it
    combined['Mass'] = combined['Mass'].round(4)
    combined = combined.groupby('Mass', as_index=False).agg({
        'Observed UL': 'min',
        'Median UL':   'min',
        'Lower 1σ':    'min',
        'Upper 1σ':    'min',
        'Lower 2σ':    'min',
        'Upper 2σ':    'min'
    })
    out_name = 'combined_UL_brazil.csv'
    combined.to_csv(out_name, index=False)
    print(f"Wrote combined table to {out_name} with {len(combined)} unique mass points.")


make_table failed at center 0.033 GeV: lam < 0 or lam contains NaNs


/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4620: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


make_table failed at center 0.035 GeV: lam < 0 or lam contains NaNs


/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:659: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4620: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4620: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4620: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/Users/aidanhsu/Documents/gaus-proc/.venv/l

Combining 147 files...
Wrote combined table to combined_UL_brazil.csv with 174 unique mass points.


/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4620: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/var/folders/lh/5k1xk47s7q9dgdbf5kby_6_h0000gn/T/ipykernel_58941/2168096242.py:161: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(df_list, ignore_index=True)


In [2]:
# … after you’ve done:
h = combined.groupby('Mass', as_index=False).min()

# keep only 0.033 ≤ Mass ≤ 0.179
mask = (combined['Mass'] >= 0.033) & (combined['Mass'] <= 0.179)
h = combined.loc[mask]

h.to_csv('combined_UL_brazil_range.csv', index=False)
